# KTPA daily climate data
Download daily temperature and precipitation observations from NOAA NCEI's Climate Data Online (CDO) API for September 29–October 9, 2026 (inclusive).

**Note:** This uses the `GHCND` daily summaries dataset.

In [61]:
import requests
import pandas as pd
import datetime as dt

In [62]:
GHCN_station_list_url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
# read the station list into a DataFrame
stations_df = pd.read_fwf(GHCN_station_list_url, header=None, widths=[11, 9, 10, 7, 3, 31, 4, 4], names=["stationid", "latitude", "longitude", "elevation", "state", "name", "gsn_flag", "hcn_crn_flag"])
stations_df

,stationid,latitude,longitude,elevation,state,name,gsn_flag,hcn_crn_flag
0,ACW00011604,17.1167,-61.7833,10.1,NaN,ST JOHNS COOLIDGE FLD,NaN,NaN
1,ACW00011647,17.1333,-61.7833,19.2,NaN,ST JOHNS,NaN,NaN
2,AE000041196,25.3330,55.5170,34.0,NaN,SHARJAH INTER. AIRP,GSN,NaN
3,AEM00041194,25.2550,55.3640,10.4,NaN,DUBAI INTL,NaN,NaN
4,AEM00041217,24.4330,54.6510,26.8,NaN,ABU DHABI INTL,NaN,NaN
...,...,...,...,...,...,...,...,...
133034,ZI000067969,-21.0500,29.3670,861.0,NaN,WEST NICHOLSON,NaN,NaN
133035,ZI000067975,-20.0670,30.8670,1095.0,NaN,MASVINGO,NaN,NaN
133036,ZI000067977,-21.0170,31.5830,430.0,NaN,BUFFALO RANGE,NaN,NaN
133037,ZI000067983,-20.2000,32.6160,1132.0,NaN,CHIPINGE,GSN,NaN


In [73]:
# We can filter the DataFrame to find stations in Washington (WA) that have "OLY" in their name, which likely refers to Olympia. 
# Here's how you can do that:
stations_df[(stations_df['state'] == 'WA') & (stations_df['name'].str.contains(r'OLY.*AP', case=False, regex=True))]

,stationid,latitude,longitude,elevation,state,name,gsn_flag,hcn_crn_flag
131407,USW00024227,46.9736,-122.905,61.0,WA,OLYMPIA AP,GSN,NaN


In [78]:
# NEED TO REMOVE THIS!!
TOKEN = "HIGCctiZzxjbzfAEiKmwZrXuihIzzLBM"

STATION = "GHCND:USW00024227"  # Tampa International Airport (KTPA)
START_DATE = "2026-09-29"
END_DATE = "2026-10-13"

In [79]:
# Show the available datatypes for the GHCND dataset and the specified station.
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/datatypes"

params = {
    "datasetid": "GHCND",
    "stationid": STATION,
    "limit": 1000
}

response = requests.get(
    url,
    headers={"token": TOKEN},
    params=params,
    timeout=30
)
response.raise_for_status()

datatypes = pd.DataFrame(response.json()["results"])

display(datatypes[["id", "name"]])

,id,name
0,ACMH,Average cloudiness midnight to midnight from m...
1,ACSH,Average cloudiness sunrise to sunset from manu...
2,AWND,Average wind speed
3,FMTM,Time of fastest mile or fastest 1-minute wind
4,FRGT,Top of frozen ground layer
5,PGTM,Peak gust time
6,PRCP,Precipitation
7,SNOW,Snowfall
8,SNWD,Snow depth
9,TAVG,Average Temperature.


In [80]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
params = {
    "datasetid": "GHCND",
    "stationid": STATION,
    "startdate": START_DATE,
    "enddate": END_DATE,
    "datatypeid": "TMAX,TMIN,WSF2,PRCP",
    "units": "standard",
    "limit": 1000,
}

if TOKEN == "PASTE_YOUR_NCEI_TOKEN_HERE":
    raise ValueError("Paste your NCEI token into the TOKEN variable first.")

response = requests.get(url, headers={"token": TOKEN}, params=params, timeout=30)
response.raise_for_status()
records = response.json().get("results", [])
print(f"Downloaded {len(records)} records.")

Downloaded 44 records.


In [81]:
if not records:
    print("No records returned. Check station availability, dates, and NCEI processing delays.")
    df = pd.DataFrame()
else:
    raw = pd.DataFrame(records)
    df = raw.pivot_table(index="date", columns="datatype", values="value", aggfunc="first")
    df.index = pd.to_datetime(df.index).date
    df.index.name = "date"
    df = df.sort_index()
    df['Wind (knots)'] = round(df['WSF2'] / 1.151, 1)  # Convert mph to knots
    display(df)

# Standard units: TMAX/TMIN in degrees F; PRCP in inches.

datatype,PRCP,TMAX,TMIN,WSF2,Wind (knots)
date,,,,,
2026-09-29,0.22,61.0,51.0,17.0,14.8
2026-09-30,0.00,66.0,45.0,13.0,11.3
2026-10-01,0.00,71.0,43.0,6.9,6.0
2026-10-02,0.00,73.0,46.0,12.1,10.5
2026-10-03,0.00,70.0,46.0,14.1,12.3
2026-10-04,0.00,72.0,43.0,8.1,7.0
2026-10-05,0.00,70.0,46.0,6.0,5.2
2026-10-06,0.00,68.0,50.0,12.1,10.5
2026-10-07,0.00,68.0,47.0,16.1,14.0


In [82]:
wx_challenge_vars = ['TMAX', 'TMIN', 'Wind (knots)', 'PRCP']

In [69]:
# create a list of specific dates that are used in the WX challenge.
# Tuesday, Wednesday, Thursday, Friday of each weak. 
start_date = dt.date(2026, 9, 29)  # Example start date
end_date = dt.date(2026, 10, 31)   # Example end date

# 1 = Tuesday, 2 = Wednesday, 3 = Thursday, 4 = Friday
# In Python's date.weekday(): Monday is 0, Tuesday is 1, ..., Sunday is 6
tue_to_fri_dates = [
    start_date + dt.timedelta(days=i)
    for i in range((end_date - start_date).days + 1)
    if 1 <= (start_date + dt.timedelta(days=i)).weekday() <= 4
]

print(tue_to_fri_dates)

[datetime.date(2026, 9, 29), datetime.date(2026, 9, 30), datetime.date(2026, 10, 1), datetime.date(2026, 10, 2), datetime.date(2026, 10, 6), datetime.date(2026, 10, 7), datetime.date(2026, 10, 8), datetime.date(2026, 10, 9), datetime.date(2026, 10, 13), datetime.date(2026, 10, 14), datetime.date(2026, 10, 15), datetime.date(2026, 10, 16), datetime.date(2026, 10, 20), datetime.date(2026, 10, 21), datetime.date(2026, 10, 22), datetime.date(2026, 10, 23), datetime.date(2026, 10, 27), datetime.date(2026, 10, 28), datetime.date(2026, 10, 29), datetime.date(2026, 10, 30)]


In [70]:
df_filtered = df[df.index.isin(tue_to_fri_dates)][wx_challenge_vars]
df_filtered

datatype,TMAX,TMIN,Wind (knots),PRCP
date,,,,
2026-09-29,87.0,74.0,10.5,0.00
2026-09-30,88.0,75.0,14.0,0.42
2026-10-01,88.0,74.0,14.0,0.28
2026-10-02,87.0,76.0,10.5,0.00
2026-10-06,79.0,76.0,7.7,1.36
2026-10-07,83.0,73.0,22.5,2.16
2026-10-08,79.0,75.0,8.8,0.66
2026-10-09,90.0,75.0,14.8,0.00


In [71]:
# To save as CSV, uncomment the following lines. The filename will include the start and end dates.
# if not df.empty:
#     filename = f"KTPA_daily_{START_DATE}_{END_DATE}.csv"
#     df.to_csv(filename)
#     print(f"Saved {filename}")